# This notebook is a foundation of the data processing script that cleans and prepares the data for further modeling

It consists of:
- standard cleaning procedures (removing duplicates, NaNs)
- types standardization
- segmenting (based on a timegap)
- creating label attribute
- speed check (unrealistic distance jumps)
- encoding COG column (from degrees to sin and cos values)
- interpolating observations so the time interval is uniform across the full dataset
- converting latitude and longitude to a UTM 32 Zone CRS
- adding derivatives as additional motion descriptive features
- acceleration and turning check (unrealistic movements)
- for one of the versions of the dataset: choosing randomly the segments to balance the classes.
- at the end the dataset is saved as pyarrow parquet files and csv

This notebook processes only one day of data for the purpose of building this cleaning pipeline, but the full data is processed with the file processing_data_script.py.

For the modeling part, three different datasets were prepared, so the different operations in this notebook were used in different combinations.

The overview of the created datasets:

1. A dataset with geographical coordinates and irregular time intervals (no interpolation). Code parts: 1-4, 11
2. A dataset with geographical coordinates, cartesian coordinates, uniform time intervals and derivatives as additional features. Code parts: 1-9, 11
3. Same as the second dataset, but for the observations with not fishing status the cargo vessels were taken. Code parts: 1-7, 9-11

___

In [ ]:
# imports
import pandas as pd
import pandas
import pyarrow
import pyarrow.parquet
import numpy as np
import os
import utm
from pyproj import Transformer

## First, we look at the raw dataframe to understand the data

In [68]:
df1 = pd.read_csv('aisdk-2022-04/aisdk-2022-04-01.csv')
df1.head()

,# Timestamp,Type of mobile,MMSI,Latitude,Longitude,Navigational status,ROT,SOG,COG,Heading,...,Length,Type of position fixing device,Draught,Destination,ETA,Data source type,A,B,C,D
0,01/04/2022 00:00:00,Class A,219423000,54.566117,11.928883,Under way using engine,-25.7,2.7,189.9,184.0,...,NaN,Undefined,NaN,NaN,NaN,AIS,NaN,NaN,NaN,NaN
1,01/04/2022 00:00:00,Class A,246553000,54.695920,12.587940,Under way using engine,1.1,9.1,56.7,58.0,...,NaN,Undefined,NaN,NaN,NaN,AIS,NaN,NaN,NaN,NaN
2,01/04/2022 00:00:00,Class A,244085000,54.694222,12.444730,Under way using engine,-8.7,13.0,235.0,235.0,...,NaN,Undefined,NaN,NaN,NaN,AIS,NaN,NaN,NaN,NaN
3,01/04/2022 00:00:00,Class B,219027620,91.000000,0.000000,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Undefined,NaN,NaN,NaN,AIS,NaN,NaN,NaN,NaN
4,01/04/2022 00:00:00,Class A,219022256,54.568840,11.930807,Under way using engine,NaN,18.8,309.1,312.0,...,NaN,Undefined,NaN,NaN,NaN,AIS,NaN,NaN,NaN,NaN


In [69]:
df1['Navigational status'].value_counts()

Navigational status
Under way using engine                                   6118366
Unknown value                                            1663483
Engaged in fishing                                       1251169
Restricted maneuverability                                449770
Moored                                                    365430
Under way sailing                                         113739
Reserved for future amendment [HSC]                        87848
Constrained by her draught                                 73495
At anchor                                                  43700
Power-driven vessel pushing ahead or towing alongside      12495
Power-driven vessel towing astern                          10795
Not under command                                           8177
Reserved for future amendment [WIG]                         2861
Reserved for future use                                        7
Name: count, dtype: int64

A few columns that should help us identify the relevant vessels (fishing ones).

In [70]:
df1[['Navigational status', 'Type of mobile', 'Ship type', 'Cargo type']].head(10)

,Navigational status,Type of mobile,Ship type,Cargo type
0,Under way using engine,Class A,Undefined,NaN
1,Under way using engine,Class A,Undefined,NaN
2,Under way using engine,Class A,Undefined,NaN
3,Unknown value,Class B,Undefined,NaN
4,Under way using engine,Class A,Undefined,NaN
5,Under way using engine,Class A,Undefined,NaN
6,Unknown value,AtoN,Undefined,NaN
7,Under way using engine,Class A,Undefined,NaN
8,Under way using engine,Class A,Undefined,NaN
9,Unknown value,Base Station,Undefined,NaN


In [71]:
df1['Ship type'].value_counts()

Ship type
Fishing                  2323350
Cargo                    2033759
Passenger                1101568
Undefined                 813129
Tanker                    766690
Other                     511553
Tug                       453450
SAR                       410962
Pilot                     372616
Dredging                  334630
HSC                       230768
Pleasure                  206527
Sailing                   174152
Military                  142321
Towing                     74190
Reserved                   61393
Law enforcement            59830
Port tender                33342
Towing long/wide           26679
Diving                     19106
Not party to conflict      17215
Anti-pollution             16755
Spare 1                     8908
Medical                     8442
Name: count, dtype: int64

What ship types have at least one status as 'Engaged in fishing'?

In [ ]:
df1[df1['Navigational status'] == 'Engaged in fishing'].groupby('Ship type').size()

Ship type
Fishing      1199476
Other          14118
Reserved        3655
Undefined      33920
dtype: int64

**We need to consider all vessels with types: Fishing, Other, Reserved, Undefined, as they all can be fishing vessels**

In [73]:
df1.isna().sum()

# Timestamp                             0
Type of mobile                          0
MMSI                                    0
Latitude                                0
Longitude                               0
Navigational status                     0
ROT                               3231131
SOG                                665290
COG                               1287692
Heading                           2234882
IMO                                     0
Callsign                           683272
Name                               559077
Ship type                               0
Cargo type                        8474744
Width                              919592
Length                             915786
Type of position fixing device          0
Draught                           2701349
Destination                       2776104
ETA                               3862422
Data source type                        0
A                                  929893
B                                 

How many rows there are if we take only the ones where ship type is in ["Fishing", "Other", "Reserved", "Undefined"] and the Navigational status is 'enagaged in fishing'?

In [ ]:
df_tryout = df1[(df1['Ship type'].isin(['Fishing', 'Other', 'Reserved', 'Undefined'])) & (df1['Navigational status'] == 'Engaged in fishing')]
df_tryout2 = df1[(df1['Ship type'].isin(['Cargo'])) & (df1['Navigational status'] == 'Engaged in fishing')]

In [ ]:
df1[df1['Ship type'] == 'Cargo']['Type of mobile'].value_counts()

Type of mobile
Class A    1074928
Class B      26640
Name: count, dtype: int64

## Processing

### 1. Standard cleaning procedures (removing duplicates, NaNs), types standardization, segmenting

The function fn() mainly consist of the pre-processing code provided by the project supervisor with a few customizations.

In [ ]:
def fn(file_path):
    dtypes = {
        "MMSI": "object",
        "SOG": float,
        "COG": float,
        "Longitude": float,
        "Latitude": float,
        "Ship type": "object",
        "# Timestamp": "object",
        "Type of mobile": "object",
        "Navigational status": "object",
    }
    usecols = list(dtypes.keys())
    df = pandas.read_csv(file_path, usecols=usecols, dtype=dtypes)

    # Remove errors
    bbox = [60,  0, 50, 20] # north, west, south, east
    north, west, south, east = bbox
    df = df[(df["Latitude"] <= north) & (df["Latitude"] >= south) & (df["Longitude"] >= west) & (
            df["Longitude"] <= east)]
    
    # # Take only the vessels with relevant ship types
    # df = df[df["Ship type"].isin(["Fishing", "Other", "Reserved", "Undefined"])].drop(columns=["Ship type"])

    # Another version of the dataset: for fishing label as true we take only the vessels with navigational status 'engaged in fishing'
    # For the fishing label as false we take Passenger or Cargo ship type to clearly distinct the two classes
    df_true_fishing = df[(df['Ship type'].isin(['Fishing', 'Other', 'Reserved', 'Undefined'])) & (df['Navigational status'] == 'Engaged in fishing')]
    df = pandas.concat([df_true_fishing, df[df['Ship type'] == 'Cargo']])

    print(df['Ship type'].value_counts())
    df = df.drop(columns=['Ship type'])


    # Drop records with Nan values
    df = df.dropna()

    # Only records from class A and class B vessels
    df = df[df["Type of mobile"].isin(["Class A", "Class B"])].drop(columns=["Type of mobile"])
    df = df[df["MMSI"].str.len() == 9]  # Adhere to MMSI format
    df = df[df["MMSI"].str[:3].astype(int).between(200, 775)]  # Adhere to MID standard

    df = df.rename(columns={"# Timestamp": "Timestamp"})
    df["Timestamp"] = pandas.to_datetime(df["Timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

    df = df.drop_duplicates(["Timestamp", "MMSI", ], keep="first")

    def track_filter(g):
        len_filt = len(g) > 256  # Min required length of track/segment
        sog_filt = 1 <= g["SOG"].max() <= 50  # Remove stationary tracks/segments
        time_filt = (g["Timestamp"].max() - g["Timestamp"].min()).total_seconds() >= 60 * 60  # Min required timespan
        return len_filt and sog_filt and time_filt

    # Track filtering
    df = df.groupby("MMSI").filter(track_filter)
    df = df.sort_values(['MMSI', 'Timestamp'])

    # Divide track into segments based on timegap
    df['Segment'] = df.groupby('MMSI')['Timestamp'].transform(
        lambda x: (x.diff().dt.total_seconds().fillna(0) >= 15 * 60).cumsum())  # Max allowed timegap

    # Segment filtering
    df = df.groupby(["MMSI", "Segment"]).filter(track_filter)
    df = df.reset_index(drop=True)

    # Units from knots to m/s
    knots_to_ms = 0.514444
    df["SOG"] = knots_to_ms * df["SOG"]

    return df

df = fn('aisdk-2022-04/aisdk-2022-04-01.csv')

Ship type
Cargo        2033534
Fishing      1195621
Undefined      33899
Other          14118
Reserved        3655
Name: count, dtype: int64


In [77]:
print('Number of unique ships:', df['MMSI'].nunique())
print('Latitude max:', df['Latitude'].max(), 'Latitude min:', df['Latitude'].min())
print('Longitude max:', df['Longitude'].max(), 'Longitude min:', df['Longitude'].min())

Number of unique ships: 508
Latitude max: 58.9516 Latitude min: 53.52997
Longitude max: 16.4744 Longitude min: 1.577742


### 2. Label attribute

As we prepare the dataset for a classification modelling, we need a label column - Fishing that will be based on the values of the Navigational Status. It will take 0 when the status is not Engaged in fishing and 1 when it is.

In [78]:
df_label = df.copy()
# create a column called Fishing that is 1 when Navigational status is 'Engaged in fishing' and 0 otherwise
df_label['Fishing'] = df_label['Navigational status'].apply(lambda x: 1 if x == 'Engaged in fishing' else 0)
df_label['Fishing'].value_counts()

Fishing
0    1874074
1     709709
Name: count, dtype: int64

In [79]:
df_label.drop(columns=['Navigational status'], inplace=True)

In [80]:
df_label

,Timestamp,MMSI,Latitude,Longitude,SOG,COG,Segment,Fishing
0,2022-04-01 00:00:55,209014000,55.024443,14.469407,4.218441,120.9,0,0
1,2022-04-01 00:01:01,209014000,55.024243,14.469987,4.166996,120.5,0,0
2,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,120.5,0,0
3,2022-04-01 00:01:32,209014000,55.023635,14.471748,4.218441,120.9,0,0
4,2022-04-01 00:01:41,209014000,55.023458,14.472258,4.218441,120.6,0,0
...,...,...,...,...,...,...,...,...
2583778,2022-04-01 01:29:18,636092983,55.521187,6.809928,8.076771,199.0,0,0
2583779,2022-04-01 01:29:54,636092983,55.518740,6.808378,8.025326,201.0,0,0
2583780,2022-04-01 01:36:24,636092983,55.492207,6.791243,8.076771,200.0,0,0
2583781,2022-04-01 01:38:07,636092983,55.485293,6.786835,7.973882,200.0,0,0


### 3. Speed check

The code below filters the records with unrealistic movements by checkin if a vessel does not do big distance jumps. It is performed by calculating the distance deltas (distance as haversine distance) and time deltas between the observations in one sequence, then by calculating mean speed. Only the records with speed lower than maximum speed = 142m/s will be kept.

In [81]:
# Give the distance in meters between two lat/lon points
def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2.0)**2
    return 2*R*np.arcsin(np.sqrt(a))

In [82]:
# Initialisation 
dataset_detect_anomaly = df_label.copy()
dataset_detect_anomaly = dataset_detect_anomaly.dropna()
dataset_detect_anomaly = dataset_detect_anomaly.sort_values(['MMSI', 'Timestamp'])

# Shift columns to compare with the previous point
dataset_detect_anomaly['lat_prev'] = dataset_detect_anomaly.groupby(['MMSI','Segment'])['Latitude'].shift(1)
dataset_detect_anomaly['lon_prev'] = dataset_detect_anomaly.groupby(['MMSI','Segment'])['Longitude'].shift(1)
dataset_detect_anomaly['time_prev'] = dataset_detect_anomaly.groupby(['MMSI','Segment'])['Timestamp'].shift(1)

# Compute distance and time vectorized between two data points
dataset_detect_anomaly['dist_m'] = haversine_vec(dataset_detect_anomaly['Latitude'], dataset_detect_anomaly['Longitude'], dataset_detect_anomaly['lat_prev'], dataset_detect_anomaly['lon_prev'])
dataset_detect_anomaly['dt_s'] = (dataset_detect_anomaly['Timestamp'] - dataset_detect_anomaly['time_prev']).dt.total_seconds()

dataset_detect_anomaly['mean_speed'] = dataset_detect_anomaly['dist_m'] / dataset_detect_anomaly['dt_s']

# Remove NaN values and first entries without previous point
dataset_detect_anomaly = dataset_detect_anomaly.dropna()

In [83]:
def clean_speed(df, vmax=142):
    if max(df['mean_speed']) > vmax:
    
        df = df[df['mean_speed'] <= vmax].copy()
        df = df.sort_values(['MMSI', 'Timestamp'])
        
        df['lat_prev'] = df.groupby(['MMSI','Segment'])['Latitude'].shift(1)
        df['lon_prev'] = df.groupby(['MMSI','Segment'])['Longitude'].shift(1)
        df['time_prev'] = df.groupby(['MMSI','Segment'])['Timestamp'].shift(1)
        
        df['dist_m'] = haversine_vec(df['Latitude'], df['Longitude'], df['lat_prev'], df['lon_prev'])
        df['dt_s'] = (df['Timestamp'] - df['time_prev']).dt.total_seconds()
        df['mean_speed'] = df['dist_m'] / df['dt_s']

    return df.dropna(subset=['mean_speed']).reset_index(drop=True)

count = 0
speed_check_df = dataset_detect_anomaly.copy()
while speed_check_df['mean_speed'].max() > 142:
    speed_check_df = clean_speed(speed_check_df, vmax=142)
    count += 1
    print(count, 'rounds completed.')

1 rounds completed.


In [84]:
speed_check_df

,Timestamp,MMSI,Latitude,Longitude,SOG,COG,Segment,Fishing,lat_prev,lon_prev,time_prev,dist_m,dt_s,mean_speed
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,120.5,0,0,55.024243,14.469987,2022-04-01 00:01:01,50.774900,12.0,4.231242
1,2022-04-01 00:01:32,209014000,55.023635,14.471748,4.218441,120.9,0,0,55.024008,14.470670,2022-04-01 00:01:13,80.260016,19.0,4.224211
2,2022-04-01 00:01:41,209014000,55.023458,14.472258,4.218441,120.6,0,0,55.023635,14.471748,2022-04-01 00:01:32,38.001808,9.0,4.222423
3,2022-04-01 00:01:51,209014000,55.023265,14.472813,4.218441,120.3,0,0,55.023458,14.472258,2022-04-01 00:01:41,41.377083,10.0,4.137708
4,2022-04-01 00:02:02,209014000,55.023053,14.473437,4.218441,120.7,0,0,55.023265,14.472813,2022-04-01 00:01:51,46.235823,11.0,4.203257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2582519,2022-04-01 01:29:18,636092983,55.521187,6.809928,8.076771,199.0,0,0,55.523153,6.811183,2022-04-01 01:28:49,232.444753,29.0,8.015336
2582520,2022-04-01 01:29:54,636092983,55.518740,6.808378,8.025326,201.0,0,0,55.521187,6.809928,2022-04-01 01:29:18,289.059507,36.0,8.029431
2582521,2022-04-01 01:36:24,636092983,55.492207,6.791243,8.076771,200.0,0,0,55.518740,6.808378,2022-04-01 01:29:54,3141.464527,390.0,8.055037
2582522,2022-04-01 01:38:07,636092983,55.485293,6.786835,7.973882,200.0,0,0,55.492207,6.791243,2022-04-01 01:36:24,817.419327,103.0,7.936110


In [85]:
print('Max speed after cleaning:', speed_check_df['mean_speed'].max())

Max speed after cleaning: 141.38263572629677


### 4. Split COG into sinCOG and cosCOG

In [86]:
cog_df = speed_check_df.copy()
cog_df.drop(columns= ['lat_prev', 'lon_prev', 'time_prev', 'dist_m', 'dt_s', 'mean_speed'], inplace=True)
cog_df

,Timestamp,MMSI,Latitude,Longitude,SOG,COG,Segment,Fishing
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,120.5,0,0
1,2022-04-01 00:01:32,209014000,55.023635,14.471748,4.218441,120.9,0,0
2,2022-04-01 00:01:41,209014000,55.023458,14.472258,4.218441,120.6,0,0
3,2022-04-01 00:01:51,209014000,55.023265,14.472813,4.218441,120.3,0,0
4,2022-04-01 00:02:02,209014000,55.023053,14.473437,4.218441,120.7,0,0
...,...,...,...,...,...,...,...,...
2582519,2022-04-01 01:29:18,636092983,55.521187,6.809928,8.076771,199.0,0,0
2582520,2022-04-01 01:29:54,636092983,55.518740,6.808378,8.025326,201.0,0,0
2582521,2022-04-01 01:36:24,636092983,55.492207,6.791243,8.076771,200.0,0,0
2582522,2022-04-01 01:38:07,636092983,55.485293,6.786835,7.973882,200.0,0,0


In [87]:
print(cog_df['COG'].min(), cog_df['COG'].max())

0.0 359.9


In [88]:
# converted COG values to sin and cos, so its real value can be reconstructed after any modifications
cog_df["sinCOG"] = np.sin(np.radians(cog_df["COG"]))
cog_df["cosCOG"] = np.cos(np.radians(cog_df["COG"]))
cog_df.drop(columns=['COG'], inplace=True)
cog_df

,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538
1,2022-04-01 00:01:32,209014000,55.023635,14.471748,4.218441,0,0,0.858065,-0.513541
2,2022-04-01 00:01:41,209014000,55.023458,14.472258,4.218441,0,0,0.860742,-0.509041
3,2022-04-01 00:01:51,209014000,55.023265,14.472813,4.218441,0,0,0.863396,-0.504528
4,2022-04-01 00:02:02,209014000,55.023053,14.473437,4.218441,0,0,0.859852,-0.510543
...,...,...,...,...,...,...,...,...,...
2582519,2022-04-01 01:29:18,636092983,55.521187,6.809928,8.076771,0,0,-0.325568,-0.945519
2582520,2022-04-01 01:29:54,636092983,55.518740,6.808378,8.025326,0,0,-0.358368,-0.933580
2582521,2022-04-01 01:36:24,636092983,55.492207,6.791243,8.076771,0,0,-0.342020,-0.939693
2582522,2022-04-01 01:38:07,636092983,55.485293,6.786835,7.973882,0,0,-0.342020,-0.939693


In [89]:
clean_df = cog_df.copy()
clean_df

,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538
1,2022-04-01 00:01:32,209014000,55.023635,14.471748,4.218441,0,0,0.858065,-0.513541
2,2022-04-01 00:01:41,209014000,55.023458,14.472258,4.218441,0,0,0.860742,-0.509041
3,2022-04-01 00:01:51,209014000,55.023265,14.472813,4.218441,0,0,0.863396,-0.504528
4,2022-04-01 00:02:02,209014000,55.023053,14.473437,4.218441,0,0,0.859852,-0.510543
...,...,...,...,...,...,...,...,...,...
2582519,2022-04-01 01:29:18,636092983,55.521187,6.809928,8.076771,0,0,-0.325568,-0.945519
2582520,2022-04-01 01:29:54,636092983,55.518740,6.808378,8.025326,0,0,-0.358368,-0.933580
2582521,2022-04-01 01:36:24,636092983,55.492207,6.791243,8.076771,0,0,-0.342020,-0.939693
2582522,2022-04-01 01:38:07,636092983,55.485293,6.786835,7.973882,0,0,-0.342020,-0.939693


In [90]:
clean_df['Fishing'].value_counts()

Fishing
0    1873069
1     709455
Name: count, dtype: int64

### 5. OPTIONAL: Interpolation of timestamps for uniform time intervals

A simple recurrent neural network is build in a way that takes a state at time t, as an input to the hidden unit. It based on the assumption that time step t is always the same, and that s(t) and s(t-1) mean always the same independent on the number of the hidden layer. Altough the implemented in the modeling part LSTM model can learn from irregular sequences, it was decided to try to interpolate and see how the model performs.

Currently, the timestamps vary from 10 seconds to 3-4 minutes, so in order to prepare the data for RNN modeling, we make uniform time intervals using interpolation between the given datapoints.

Additionally, the interpolation is done in two ways: using spherical calculation (slerp) for the coordinates, and linear for the rest of the columns.

In [91]:
# average timegap between observations that have the same mmsi and the same segment
time_diffs = clean_df.sort_values(['MMSI', 'Segment', 'Timestamp']).groupby(['MMSI', 'Segment'])['Timestamp'].diff().dt.total_seconds()
average_timegap = time_diffs.mean()
print('Average timegap between observations:', average_timegap, 'seconds')

Average timegap between observations: 10.726776798623357 seconds


Based on the average timegap between observations, the time interval will be set to be 12 seconds and the observations will be interpolated.

- dt = 12 for the first dataset with geographical coordinates and for the second with cartesian attributes and uniform time intervals
- dt = 11 for the dataset with mixed fishing boats and cargo

In [92]:
# dt = 12  # time interval in seconds after interpolation
dt = 10

**INTERPOLATION FUNCTIONS**

Disclaimer: interpolation functions are created with a considerable AI help.

In [93]:
# two functions converting coordinates from lat/lon to spherical and back from spherical to lat/lon

def latlon_to_xyz(lat, lon):
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    x = np.cos(lat_r) * np.cos(lon_r)
    y = np.cos(lat_r) * np.sin(lon_r)
    z = np.sin(lat_r)
    return np.vstack([x, y, z]).T

def xyz_to_latlon(xyz):
    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    lat = np.degrees(np.arctan2(z, np.sqrt(x*x + y*y)))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

In [94]:
# computing the spherical interpolation
def slerp(v0, v1, t):
    dot = np.clip(np.sum(v0 * v1, axis=1), -1.0, 1.0)
    theta = np.arccos(dot)

    # Avoid division by zero for identical vectors
    sin_theta = np.sin(theta)
    small = sin_theta < 1e-8

    out = np.zeros_like(v0)
    w0 = np.zeros_like(theta)
    w1 = np.zeros_like(theta)

    # compute only where valid
    valid = ~small
    w0[valid] = np.sin((1 - t[valid]) * theta[valid]) / sin_theta[valid]
    w1[valid] = np.sin(t[valid] * theta[valid]) / sin_theta[valid]
    
    out[valid] = w0[valid, None] * v0[valid] + w1[valid, None] * v1[valid]

    # If the points are identical, just copy v0
    out[small] = v0[small]
    return out

In [95]:
# a full resample (interpolation of an observation) for one segment
def make_uniform_time_slerp(df_segment, dt):

    df = df_segment.set_index("Timestamp")
    t_min, t_max = df.index.min(), df.index.max()
    new_index = pd.date_range(t_min, t_max, freq=f"{dt}s")

    df_reindexed = df.reindex(new_index)

    # linear interpolation of all numeric columns except Fishing
    numeric_cols = df.select_dtypes(include=["number"]).columns.drop(["Latitude","Longitude", "Fishing"])
    df_reindexed[numeric_cols] = (
        df_reindexed[numeric_cols]
        .interpolate(method="linear", limit_direction="both")
    )

    # forward fill the Fishing column to maintain the binary label
    df_reindexed["Fishing"] = df_reindexed["Fishing"].ffill().bfill()
    
    # slerp calculated for the coordinates
    t_orig = df.index.values.astype("datetime64[ns]").astype(float)
    t_new = df_reindexed.index.values.astype("datetime64[ns]").astype(float)

    lat = df["Latitude"].values
    lon = df["Longitude"].values

    xyz = latlon_to_xyz(lat, lon)

    # the indices of the interpolated observations should be in range, and it the correct order/place
    idx = np.searchsorted(t_orig, t_new)
    idx = np.clip(idx, 1, len(t_orig)-1)

    t0 = t_orig[idx-1]
    t1 = t_orig[idx]
    v0 = xyz[idx-1]
    v1 = xyz[idx]

    tt = (t_new - t0) / (t1 - t0)

    xyz_interp = slerp(v0, v1, tt)
    lat_interp, lon_interp = xyz_to_latlon(xyz_interp)

    df_reindexed["Latitude"] = lat_interp
    df_reindexed["Longitude"] = lon_interp

    # fill-in the columns that were not interpolated - MMSI, Segment, Date and Fishing
    constant_cols = ["MMSI", "Segment", "Date"]
    for col in constant_cols:
        if col in df_segment.columns:
            df_reindexed[col] = df_segment[col].iloc[0]

    return df_reindexed.reset_index().rename(columns={"index": "Timestamp"})


# a function that computes interpolation to a whole dataset
def uniform_resample_all_slerp(df, dt):
    return (df.groupby(["MMSI", "Segment"], group_keys=False).apply(lambda g: make_uniform_time_slerp(g, dt)).reset_index(drop=True))


In [96]:
# INTERPOLATION
uniform_dataset = uniform_resample_all_slerp(clean_df, dt)

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_9929/484204554.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return (df.groupby(["MMSI", "Segment"], group_keys=False).apply(lambda g: make_uniform_time_slerp(g, dt)).reset_index(drop=True))


In [97]:
# convert Fishing column to integer type
uniform_dataset['Fishing'] = uniform_dataset['Fishing'].astype(int)
uniform_dataset['Fishing'].value_counts()

Fishing
0    2066004
1     703935
Name: count, dtype: int64

COG can be recreated after interpolation to a form more suited for human interpretation (degrees), but cannot be used for modeling (0 degrees and 360 degrees).

In [98]:
# recreating COG after interpolation, for human interpretation o
# uniform_dataset['COG'] = np.degrees(np.arctan2(uniform_dataset['sinCOG'], uniform_dataset['cosCOG'])) % 360
# uniform_dataset

Interpolation can create a few nan values that need to be identified and removed from the dataset without destroying the time regularity, so we perform a data quality check.

In [99]:
# check if longitude and latitude have any zero values
print('Number of zero Latitude values:', (uniform_dataset['Latitude'] == 0).sum())
print('Number of zero Longitude values:', (uniform_dataset['Longitude'] == 0).sum())
# check if they have any inf values
print('Number of infinite Latitude values:', np.isinf(uniform_dataset['Latitude']).sum())
print('Number of infinite Longitude values:', np.isinf(uniform_dataset['Longitude']).sum())
# check if there are any NaN values
print('Number of NaN Latitude values:', uniform_dataset['Latitude'].isna().sum())
print('Number of NaN Longitude values:', uniform_dataset['Longitude'].isna().sum())

Number of zero Latitude values: 0
Number of zero Longitude values: 0
Number of infinite Latitude values: 0
Number of infinite Longitude values: 0
Number of NaN Latitude values: 0
Number of NaN Longitude values: 0


In [100]:
#check indices of the nan values
nan_lat_indices = uniform_dataset[uniform_dataset['Latitude'].isna()].index.tolist()
nan_lon_indices = uniform_dataset[uniform_dataset['Longitude'].isna()].index.tolist()
print('Indices of NaN Latitude values:', nan_lat_indices)
print('Indices of NaN Longitude values:', nan_lon_indices)

# print mmsi value for every row with nan
print('MMSI values for NaN Latitude values:', uniform_dataset.loc[nan_lat_indices, 'MMSI'].tolist())
print('MMSI values for NaN Longitude values:', uniform_dataset.loc[nan_lon_indices, 'MMSI'].tolist())

# print how many rows have every mmsi in the lists above
mmsi_with_nan = set(uniform_dataset.loc[nan_lat_indices, 'MMSI'].tolist() + uniform_dataset.loc[nan_lon_indices, 'MMSI'].tolist())
for mmsi in mmsi_with_nan:
    count = (uniform_dataset['MMSI'] == mmsi).sum()
    print(f'MMSI: {mmsi}, Number of rows in dataset: {count}')

# remove all rows with the mmsi from lists: uniform_dataset.loc[nan_lat_indices, 'MMSI'].tolist() and uniform_dataset.loc[nan_lon_indices, 'MMSI'].tolist()
df_to_utm = uniform_dataset.copy()
df_to_utm = df_to_utm[~df_to_utm['MMSI'].isin(mmsi_with_nan)].reset_index(drop=True)
df_to_utm

Indices of NaN Latitude values: []
Indices of NaN Longitude values: []
MMSI values for NaN Latitude values: []
MMSI values for NaN Longitude values: []


,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538
1,2022-04-01 00:01:23,209014000,55.023812,14.471237,4.166996,0,0,0.861184,-0.508289
2,2022-04-01 00:01:33,209014000,55.023615,14.471805,4.166996,0,0,0.860739,-0.509040
3,2022-04-01 00:01:43,209014000,55.023419,14.472369,4.166996,0,0,0.860295,-0.509791
4,2022-04-01 00:01:53,209014000,55.023226,14.472926,4.166996,0,0,0.859850,-0.510541
...,...,...,...,...,...,...,...,...,...
2769934,2022-04-01 01:38:00,636092983,55.485763,6.787135,7.973882,0,0,-0.342020,-0.939693
2769935,2022-04-01 01:38:10,636092983,55.485084,6.786701,7.973882,0,0,-0.342020,-0.939693
2769936,2022-04-01 01:38:20,636092983,55.484387,6.786254,7.973882,0,0,-0.342020,-0.939693
2769937,2022-04-01 01:38:30,636092983,55.483691,6.785808,7.973882,0,0,-0.342020,-0.939693


In [101]:
# check again for too short segments
def track_filter(g):
    len_filt = len(g) > 256  # Min required length of track/segment
    time_filt = (g["Timestamp"].max() - g["Timestamp"].min()).total_seconds() >= 60 * 60  # Min required timespan
    # add a filter that checks if segments dont have timegap larger than 15 minutes
    timegap_filt = (g["Timestamp"].diff().dt.total_seconds().fillna(0) < 15 * 60).all()
    return len_filt and time_filt and timegap_filt

df_to_utm_check = df_to_utm.copy()
df_to_utm_check = df_to_utm_check.groupby("MMSI").filter(track_filter)
df_to_utm_check = df_to_utm_check.sort_values(['MMSI', 'Timestamp'])

# how many rows were removed
rows_removed = len(df_to_utm) - len(df_to_utm_check)
print('Number of rows removed after filtering for too short segments:', rows_removed)

Number of rows removed after filtering for too short segments: 425201


In [102]:
df_to_utm_check['Fishing'].value_counts()

Fishing
0    1774098
1     570640
Name: count, dtype: int64

### 6. OPTIONAL: Conversion to cartesian metric system (UTM)

Another step is conversion of the latitude and longitude to a cartesian coordinate system. 

In [103]:
# Create a transformer to convert from WGS84 (lat/lon) to meters (UTM zone 32N - CRS 32632)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32632", always_xy=True)

utm_df = df_to_utm_check.copy()
utm_df["x"], utm_df["y"] = transformer.transform(utm_df["Longitude"].values, utm_df["Latitude"].values)
drop_cols = [c for c in ["utm_zone", "utm_letter"] if c in utm_df.columns]
utm_df = utm_df.drop(columns=drop_cols)

utm_df

,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG,x,y
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538,849559.227548,6.111154e+06
1,2022-04-01 00:01:23,209014000,55.023812,14.471237,4.166996,0,0,0.861184,-0.508289,849597.154664,6.111135e+06
2,2022-04-01 00:01:33,209014000,55.023615,14.471805,4.166996,0,0,0.860739,-0.509040,849635.077660,6.111117e+06
3,2022-04-01 00:01:43,209014000,55.023419,14.472369,4.166996,0,0,0.860295,-0.509791,849672.808250,6.111098e+06
4,2022-04-01 00:01:53,209014000,55.023226,14.472926,4.166996,0,0,0.859850,-0.510541,849710.074027,6.111079e+06
...,...,...,...,...,...,...,...,...,...,...,...
2769934,2022-04-01 01:38:00,636092983,55.485763,6.787135,7.973882,0,0,-0.342020,-0.939693,360173.946211,6.151074e+06
2769935,2022-04-01 01:38:10,636092983,55.485084,6.786701,7.973882,0,0,-0.342020,-0.939693,360144.153735,6.150999e+06
2769936,2022-04-01 01:38:20,636092983,55.484387,6.786254,7.973882,0,0,-0.342020,-0.939693,360113.470971,6.150923e+06
2769937,2022-04-01 01:38:30,636092983,55.483691,6.785808,7.973882,0,0,-0.342020,-0.939693,360082.788227,6.150846e+06


From this point we will handle two datasets:

1. latlon_df with original attributes
2. the dataset with attributes in the cartesian metric system

In [104]:
# prepare the dataset with lat/lon coordinates
latlon_df = utm_df.copy()
latlon_df.drop(columns=['x', 'y'], inplace=True)

# latlon_df["dSOG_dt"] = latlon_df.groupby(["MMSI", "Segment"])["SOG"].diff() / dt
latlon_df

,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538
1,2022-04-01 00:01:23,209014000,55.023812,14.471237,4.166996,0,0,0.861184,-0.508289
2,2022-04-01 00:01:33,209014000,55.023615,14.471805,4.166996,0,0,0.860739,-0.509040
3,2022-04-01 00:01:43,209014000,55.023419,14.472369,4.166996,0,0,0.860295,-0.509791
4,2022-04-01 00:01:53,209014000,55.023226,14.472926,4.166996,0,0,0.859850,-0.510541
...,...,...,...,...,...,...,...,...,...
2769934,2022-04-01 01:38:00,636092983,55.485763,6.787135,7.973882,0,0,-0.342020,-0.939693
2769935,2022-04-01 01:38:10,636092983,55.485084,6.786701,7.973882,0,0,-0.342020,-0.939693
2769936,2022-04-01 01:38:20,636092983,55.484387,6.786254,7.973882,0,0,-0.342020,-0.939693
2769937,2022-04-01 01:38:30,636092983,55.483691,6.785808,7.973882,0,0,-0.342020,-0.939693


### 7. OPTIONAL: Cartesian motion attributes and derivatives

We can also add more motion attributes, so the RNN learns faster vessel dynamics. It could be cartesian velocity (vx, vy), acceleration in both directions.

In [105]:
cartesian_df = utm_df.copy()

# cartesian_df.drop(columns=['Longitude', 'Latitude'], inplace=True)

cartesian_df["vx"] = cartesian_df["SOG"] * cartesian_df["cosCOG"]
cartesian_df["vy"] = cartesian_df["SOG"] * cartesian_df["sinCOG"]

# not dropping SOG, sinCOG, cosCOG as we are going with full dataset
# cartesian_df.drop(columns=['SOG', 'sinCOG', 'cosCOG'], inplace=True)

cartesian_df["ax"] = cartesian_df.groupby(["MMSI", "Segment"])["vx"].diff() / dt
cartesian_df["ay"] = cartesian_df.groupby(["MMSI", "Segment"])["vy"].diff() / dt

# replacing NaNs (first row in segment) with 0
cartesian_df[["ax", "ay"]] = cartesian_df[["ax", "ay"]].fillna(0)

In [ ]:
cartesian_df['Fishing'] = cartesian_df['Fishing'].astype(int)

In [106]:
cartesian_df

,Timestamp,MMSI,Latitude,Longitude,SOG,Segment,Fishing,sinCOG,cosCOG,x,y,vx,vy,ax,ay
0,2022-04-01 00:01:13,209014000,55.024008,14.470670,4.166996,0,0,0.861629,-0.507538,849559.227548,6.111154e+06,-2.114911,3.590406,0.000000,0.000000
1,2022-04-01 00:01:23,209014000,55.023812,14.471237,4.166996,0,0,0.861184,-0.508289,849597.154664,6.111135e+06,-2.118039,3.588552,-0.000313,-0.000185
2,2022-04-01 00:01:33,209014000,55.023615,14.471805,4.166996,0,0,0.860739,-0.509040,849635.077660,6.111117e+06,-2.121167,3.586698,-0.000313,-0.000185
3,2022-04-01 00:01:43,209014000,55.023419,14.472369,4.166996,0,0,0.860295,-0.509791,849672.808250,6.111098e+06,-2.124296,3.584844,-0.000313,-0.000185
4,2022-04-01 00:01:53,209014000,55.023226,14.472926,4.166996,0,0,0.859850,-0.510541,849710.074027,6.111079e+06,-2.127424,3.582990,-0.000313,-0.000185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2769934,2022-04-01 01:38:00,636092983,55.485763,6.787135,7.973882,0,0,-0.342020,-0.939693,360173.946211,6.151074e+06,-7.492998,-2.727228,0.000000,0.000000
2769935,2022-04-01 01:38:10,636092983,55.485084,6.786701,7.973882,0,0,-0.342020,-0.939693,360144.153735,6.150999e+06,-7.492998,-2.727228,0.000000,0.000000
2769936,2022-04-01 01:38:20,636092983,55.484387,6.786254,7.973882,0,0,-0.342020,-0.939693,360113.470971,6.150923e+06,-7.492998,-2.727228,0.000000,0.000000
2769937,2022-04-01 01:38:30,636092983,55.483691,6.785808,7.973882,0,0,-0.342020,-0.939693,360082.788227,6.150846e+06,-7.492998,-2.727228,0.000000,0.000000


### 8. OPTIONAL: Acceleration and turning check

Having the derivatives also enables checking if acceleration and turning are realistic (the thresholds are based on a literature review).

In [107]:
def flag_accel_and_turn(df, dt_seconds=30,
                        acc_thr=1.0,
                        turn_thr=0.3):

    out = df.copy()

    # acceleration magnitude
    out['a_mag'] = np.sqrt(out['ax']**2 + out['ay']**2)

    # turning rate
    d_sin = out.groupby(['MMSI','Segment'])['sinCOG'].diff()
    d_cos = out.groupby(['MMSI','Segment'])['cosCOG'].diff()

    out['turn_rate'] = np.arctan2(d_sin, d_cos) / dt_seconds

    # fill NaNs for the first row of each segment
    out[['a_mag', 'turn_rate']] = out[['a_mag', 'turn_rate']].fillna(0)

    # flags
    out['Flag'] = (out['a_mag'] > acc_thr) | (out['turn_rate'].abs() > turn_thr)

    return out


In [108]:
# flag_df = flag_accel_and_turn(cartesian_df, dt_seconds=dt) 

In [109]:
# # # count how many rows have Flag = True
# num_flagged = flag_df[flag_df['Flag'] == True].shape[0]
# print('Number of rows flagged as anomalous:', num_flagged)

In [110]:
# # # remove the segments that have any flagged rows
# flagged_segments = flag_df[flag_df['Flag'] == True][['MMSI', 'Segment']].drop_duplicates()
# clean_flag_df = flag_df.merge(flagged_segments, on=['MMSI', 'Segment'], how='left', indicator=True)
# clean_flag_df = clean_flag_df[clean_flag_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# # # drop the flag columns
# clean_flag_df = clean_flag_df.drop(columns=['a_mag', 'turn_rate', 'Flag'])

# # # how many rows were removed
# num_removed = cartesian_df.shape[0] - clean_flag_df.shape[0]
# print('Number of rows removed as anomalous:', num_removed)


### 9. Data quality check after all modifications

In [111]:
clean_flag_df = cartesian_df.copy()

# # check if numerical attributes have any zero values
print('Number of zero values:\n', (clean_flag_df[['Latitude', 'Longitude','x', 'y']] == 0).sum())
# # check if they have any inf values
print('Number of infinite values:\n', np.isinf(clean_flag_df[['Latitude', 'Longitude', 'x', 'y', 'vx', 'vy', 'ax', 'ay', 'sinCOG', 'cosCOG']]).sum())
# # check if there are any NaN values
print('Number of NaN values:\n', clean_flag_df[['Latitude', 'Longitude', 'x', 'y', 'vx', 'vy', 'ax', 'ay', 'sinCOG', 'cosCOG']].isna().sum())

df_quality_check = clean_flag_df.copy()
df_quality_check = df_quality_check.groupby("MMSI").filter(track_filter)
df_quality_check = df_quality_check.sort_values(['MMSI', 'Timestamp'])

# how many rows were removed
rows_removed = len(clean_flag_df) - len(df_quality_check)
print('Number of rows removed after filtering for too short segments:', rows_removed)

Number of zero values:
 Latitude     0
Longitude    0
x            0
y            0
dtype: int64
Number of infinite values:
 Latitude     0
Longitude    0
x            0
y            0
vx           0
vy           0
ax           0
ay           0
sinCOG       0
cosCOG       0
dtype: int64
Number of NaN values:
 Latitude     0
Longitude    0
x            0
y            0
vx           0
vy           0
ax           0
ay           0
sinCOG       0
cosCOG       0
dtype: int64
Number of rows removed after filtering for too short segments: 0


Finally, the dataset is partitioned and saved as parquet files. Each segment will be stored as a seperate file. 

In [112]:
final_df = df_quality_check.copy()

In [113]:
final_df["Fishing"].value_counts()

Fishing
0    1774098
1     570640
Name: count, dtype: int64

### 10. OPTIONAL: Random selection of the segments for balancing classes

Disclaimer: code for this function is mostly AI-generated. The idea of this step is to take random segments with not fishing label to balance out the number of observations in the classes. As the rows are ordered and considered as sequences (segments), it could not be implemented with just randomly selecting the rows.

In [ ]:
def balance_by_mmsi(
    df,
    label_col='Fishing',
    minority_label=1,
    random_state=42,
    preserve_minority=True
):
    rng = np.random.default_rng(random_state)

    # counts
    class_counts = df[label_col].value_counts()
    if minority_label not in class_counts.index:
        raise ValueError(f"minority_label {minority_label} not present in data")

    minority_count = class_counts[minority_label]
    majority_label = [c for c in class_counts.index if c != minority_label][0]
    majority_count = class_counts[majority_label]

    # If already balanced or minority bigger, return copy
    if majority_count <= minority_count:
        print("Already balanced or minority >= majority. Returning original.")
        return df.copy()

    # Build per-MMSI sizes for majority class
    maj_df = df[df[label_col] == majority_label]
    mmsi_sizes = (
        maj_df.groupby('MMSI')
        .size()
        .reset_index(name='n_rows')
    ).sort_values('n_rows', ascending=False).reset_index(drop=True)

    # Optionally ensure MMSIs that contain minority rows are preserved
    if preserve_minority:
        mmsi_with_minority = df[df[label_col] == minority_label]['MMSI'].unique()
        # All these MMSIs must be preserved if they exist in majority set -- otherwise minority-only MMSIs are unaffected
        mmsi_sizes = mmsi_sizes[~mmsi_sizes['MMSI'].isin(mmsi_with_minority)].reset_index(drop=True)

    # shuffle mmsis, then accumulate until we reach target
    mmsi_sizes_shuffled = mmsi_sizes.sample(frac=1, random_state=random_state).reset_index(drop=True)
    mmsi_sizes_shuffled['cum_rows'] = mmsi_sizes_shuffled['n_rows'].cumsum()

    # pick minimal prefix of MMSIs whose cumulative rows >= minority_count
    idx = mmsi_sizes_shuffled['cum_rows'].searchsorted(minority_count, side='left')
    if idx >= len(mmsi_sizes_shuffled):
        # can't reach target without keeping every majority-MMSI (we'll keep them all)
        chosen_mmsis = mmsi_sizes_shuffled['MMSI'].tolist()
    else:
        chosen_mmsis = mmsi_sizes_shuffled.iloc[: idx + 1]['MMSI'].tolist()

    # Selected majority rows (entire MMSIs)
    maj_selected = maj_df[maj_df['MMSI'].isin(chosen_mmsis)].copy()

    # Minority rows (keep all)
    min_df = df[df[label_col] == minority_label].copy()

    balanced = pd.concat([maj_selected, min_df], ignore_index=True)
    balanced = balanced.sort_values(['MMSI', 'Timestamp']).reset_index(drop=True)

    print("Original counts:", class_counts.to_dict())
    print("After balancing counts:", balanced[label_col].value_counts().to_dict())
    return balanced


final_df_balanced = balance_by_mmsi(final_df, label_col='Fishing', minority_label=1, random_state=42)

Original counts: {0: 1774098, 1: 570640}
After balancing counts: {0: 575481, 1: 570640}


In [119]:
final_df_balanced['Fishing'].value_counts()

Fishing
0    575481
1    570640
Name: count, dtype: int64

In [1]:
# check if the segments were kept correctly
final_df_balanced_check = final_df_balanced.copy()
final_df_balanced_check = final_df_balanced_check.groupby("MMSI").filter(track_filter)
final_df_balanced_check = final_df_balanced_check.sort_values(['MMSI', 'Timestamp'])

# how many rows were removed
rows_removed = len(final_df_balanced) - len(final_df_balanced_check)
print('Number of rows removed after filtering for too short segments:', rows_removed)

NameError: name 'final_df_balanced' is not defined

In [124]:
final_df_balanced_check['Fishing'].value_counts()

Fishing
0    575481
1    570640
Name: count, dtype: int64

### 11. Saving files

In [122]:
out_path = 'clean_data/2022-04-01'

# Save as parquet file with partitions
table = pyarrow.Table.from_pandas(final_df_balanced_check, preserve_index=False)
pyarrow.parquet.write_to_dataset(
    table,
    root_path=out_path,
    partition_cols=["MMSI",  # "Date",
                    "Segment",  # "Geocell"
                    ]
)


# also save as a csv file
final_df_balanced_check.to_csv('clean_data/2022-04-01.csv')